# Stage-2 cache fill: one shard (Week 3) — Kaggle

Generated from `cache_fill_colab.ipynb` by
`experiments_anupam/make_kaggle_notebook.py`. Only the two platform cells
differ; every scientific cell is byte-identical, which the generator asserts.

**Before running:** Settings → Accelerator → **GPU T4 x2** (one is used), and
Settings → **Internet: On** (the clone, pip and the ir_datasets download all
need it). Then Run All. The shard is written to `/kaggle/working/cache/`,
which survives the session and is downloadable from the Output tab.

If the session dies, Run All again: the cache is keyed on
`(query_id, doc_id, model)` and the fill resumes from whatever is already in
`/kaggle/working/cache/`.


## 1. Code, queries and qrels

In [ ]:
OWNER = 'anupam'           # anupam | shubham | chaitanya
BRANCH = 'main'

# /kaggle/working persists for the session and is kept as notebook output, so
# it does the job Drive does on Colab - with no mount and no authorisation.
DRIVE_DIR = '/kaggle/working/cache'

!rm -rf /kaggle/working/IRE_PROJECT
!git clone -q -b {BRANCH} https://github.com/a-n-u-p-a-m/IRE_PROJECT.git /kaggle/working/IRE_PROJECT
%cd /kaggle/working/IRE_PROJECT
!git log --oneline -1
!pip install -q ir_datasets sentence-transformers
!python -m experiments_shubham.fetch_trec_dl


## 2. Rebuild the frozen universe from the committed runs

Deterministic: it must come out at 97 queries and 872,455 pairs under fingerprint
`46d0807d8ed3`, or the shard would be scored against a different universe.

In [ ]:
!python -m experiments_anupam.data_prep --queries data/queries.tsv --qrels-txt data/qrels.txt
!python -m experiments_anupam.build_universe --qrels data/qrels.parquet

import pandas as pd
from shared.config import load_config
from shared.universe import load_frozen
cfg = load_config('config/base.yaml')
universe = load_frozen('data/universe.parquet', fingerprint=cfg.fingerprint())
assert cfg.fingerprint() == '46d0807d8ed3', cfg.fingerprint()
assert universe['query_id'].nunique() == 97 and len(universe) == 872_455, len(universe)
print('universe ok:', cfg.fingerprint(), len(universe))

## 3. This shard's queries

In [ ]:
from experiments_chaitanya.shard_plan import build_shard_plan
from experiments_chaitanya.cache_filler import load_query_texts

plan = build_shard_plan(universe, n_shards=3)
shard = next(s for s in plan['shards'] if s['owner'] == OWNER)
qids = [str(q) for q in shard['query_ids']]
expected = {'anupam': (33, 295_531), 'shubham': (32, 287_058), 'chaitanya': (32, 289_866)}
assert (shard['n_queries'], shard['n_pairs']) == expected[OWNER], shard
mine = universe[universe['query_id'].isin(qids)]
query_texts = load_query_texts('data/queries.tsv')
print(OWNER, len(qids), 'queries,', len(mine), 'pairs')

## 4. Resume a partial shard, if there is one

`/kaggle/working/cache/` survives within a session, so a notebook that died
mid-fill resumes from whatever it had written. Run All again; the cache is
keyed on `(query_id, doc_id, model)` and already-scored pairs are skipped.


In [ ]:
import os, shutil

# No drive.mount: DRIVE_DIR is already a persistent local path on Kaggle.
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs('cache', exist_ok=True)
SHARD = f'cache/stage2_shard_{OWNER}.sqlite'
saved = f'{DRIVE_DIR}/stage2_shard_{OWNER}.sqlite'
if os.path.exists(saved) and not os.path.exists(SHARD):
    shutil.copy(saved, SHARD)
    print('restored', saved)


## 5. Reranker, passages and a shuffle-invariance spot check

Frozen settings from the config. The spot check re-scores one query's first 256
passages in three shuffled orders; a pointwise reranker must give identical scores.

In [ ]:
import torch
from experiments_chaitanya.reranker import PointwiseCrossEncoder
from experiments_chaitanya.passage_fetcher import IrDatasetsPassageFetcher
from shared.cache import Stage2Cache

assert torch.cuda.is_available(), 'switch the runtime to a GPU'
print(torch.cuda.get_device_name(0))
reranker = PointwiseCrossEncoder(cfg.reranker_model, max_length=cfg.reranker_max_length,
                                 batch_size=cfg.reranker_batch_size,
                                 precision=cfg.reranker_precision)
fetcher = IrDatasetsPassageFetcher(required_doc_ids=set(mine['doc_id'].astype(str)))
cache = Stage2Cache(SHARD)
print('already cached:', cache.count(reranker.checkpoint))

q0 = qids[0]
d0 = mine[mine['query_id'] == q0]['doc_id'].astype(str).tolist()[:256]
assert reranker.verify_shuffle_invariance(query_texts[q0], fetcher.fetch_many(d0), d0)
print('shuffle invariance holds')

## 6. Fill

Eight queries per chunk, copied to Drive after each, so a disconnect loses at most one
chunk. Rerunning this cell resumes.

In [ ]:
import json, time
from experiments_chaitanya.cache_filler import CacheFiller

filler = CacheFiller(reranker=reranker, fetcher=fetcher, cache=cache)
reports = []
for i in range(0, len(qids), 8):
    rep = filler.fill_shard(universe=universe, query_ids=qids[i:i + 8], query_texts=query_texts)
    reports.append(rep)
    shutil.copy(SHARD, saved)
    print(f'{min(i + 8, len(qids))}/{len(qids)} queries |', f'{rep.pairs_per_second:.0f} pairs/s',
          '| errors:', len(rep.errors))
scored = sum(r.n_pairs_scored for r in reports)
seconds = sum(r.elapsed_seconds for r in reports)
print(f'scored {scored} pairs in {seconds / 60:.1f} min ({scored / max(seconds, 1e-9):.0f} pairs/s)')

## 7. Verify and save

Every pair of the shard must be cached, under the frozen model name. The digest is
order-independent, so it identifies the scores whichever way the shards are merged.

In [ ]:
from experiments_shubham.real import cache_digest
from shared.provenance import run_manifest, write_manifest

cov = filler.verify_completeness(universe, qids)
assert int(cov['n_missing'].sum()) == 0, cov.sort_values('n_missing').tail()
assert cache.models() == [cfg.reranker_model], cache.models()
manifest = run_manifest(cfg.fingerprint(), owner=OWNER, extra={
    'artifact': SHARD, 'shard_owner': OWNER, 'query_ids': qids,
    'n_pairs': cache.count(cfg.reranker_model), 'stage2_model': cfg.reranker_model,
    'max_length': cfg.reranker_max_length, 'batch_size': cfg.reranker_batch_size,
    'precision': cfg.reranker_precision, 'gpu': torch.cuda.get_device_name(0),
    'digest': cache_digest(cache, cfg.reranker_model),
    'pairs_scored_this_session': scored, 'seconds_this_session': round(seconds, 1),
    'errors': [e for r in reports for e in r.errors],
})
write_manifest(f'cache/stage2_shard_{OWNER}_manifest.json', manifest)
shutil.copy(SHARD, saved)
shutil.copy(f'cache/stage2_shard_{OWNER}_manifest.json', DRIVE_DIR)
print(json.dumps({k: manifest[k] for k in ('n_pairs', 'digest', 'gpu')}, indent=1))
!ls -la {DRIVE_DIR}

## 8. Cross-machine canary — post this digest

The three shards are disjoint, so no pair is ever scored twice and a numerical
difference between the machines that produced them is **silent**: each shard's
digest covers only its own pairs, and `merge_from` unions them without ever
comparing like with like.

Shard 2 was filled under `torch 2.11.0+cu130` on Python 3.13.16. Kaggle will
differ, and float32 matmul is not bit-reproducible across cuBLAS versions.

`shared/canary.py` scores eight frozen literal pairs — no corpus, no universe,
no filled shard, milliseconds. **Post `CANARY_DIGEST` in the group.** All three
must match. If they differ, `compare_canaries` says whether the merge is still
sound; a difference in *ordering* means it is not.


In [ ]:
from shared.canary import canary_digest, canary_scores

CANARY_SCORES = canary_scores(reranker)
CANARY_DIGEST = canary_digest(reranker)

for d, s in CANARY_SCORES.items():
    print(f'{d}  {s: .6f}')
print('\nCANARY_DIGEST', CANARY_DIGEST, '  <- post this')
print('torch', torch.__version__, '| gpu', torch.cuda.get_device_name(0))
